# ARC-v0.24 — MS MARCO External Boundary Replication

**Purpose.** Test whether the mechanism-dependent approximation-feedback boundary observed in the current paper transfers to a large, non-Wikipedia corpus without changing the feedback design after seeing trajectory outcomes.

This notebook is intentionally narrower than the main ARC lineage:

- dataset: **BEIR MS MARCO passage corpus**
- encoder: **`intfloat/e5-small-v2`**
- representation contrast: **IVF-PQ32 → IVF-SQ8**, both `nprobe=64`
- search-effort contrast: **IVF-SQ8 `nprobe=8 → 64`**
- same 44 anchored mean/softmax feedback policies
- same four feedback updates
- same H1 / H2 / H3abs / H3signed definitions
- primary statistical unit: **query**
- negative / null / reversal outcomes are retained

### Important provenance statement

MS MARCO has existed elsewhere in the broader retrieval/compression workspace, so this notebook does **not** call the dataset globally pristine.  
The intended claim is narrower:

> **ARC feedback-trajectory outcomes for this full-corpus E5 protocol are outcome-blind at freeze time.**

The notebook does not read old MS MARCO trajectory outcomes or use them to select the contrast.

### Why v0.24 uses IVF nprobe rather than HNSW first

The main paper already shows the search-effort reversal under both IVF nprobe and HNSW. On a new 8.8M-document corpus, using PQ/SQ8 plus nprobe lets the external replication test both representation and search-effort mechanisms **without paying for a second multi-million-document graph build**. If this result is informative and Compute Units remain, HNSW can be added later as ARC-v0.24b or as part of ARC-v0.25.

### Primary falsifiable directional expectations

These expectations are frozen before full trajectory outcomes:

- representation: `mean(H3abs) > 0`
- search effort: `mean(H3abs) < 0`

These are **hypotheses, not success requirements**. Any sign combination is retained.

### Storage strategy

Because Google Drive storage is already relatively full, large corpus embeddings and FAISS indexes default to **ephemeral `/content` storage**.  
Compact protocols, checkpoints, summaries, split files, and final reports are written to Drive.

Set `PERSIST_LARGE_ARTIFACTS=True` only if you have sufficient Drive space and want expensive embeddings/indexes to survive a runtime reset.

In [ ]:
# Cell 1 — Install / imports / execution controls
!pip -q install faiss-cpu sentence-transformers pyarrow scipy tqdm psutil

from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
import gc, hashlib, json, math, os, random, shutil, time, warnings, zipfile

import faiss
import numpy as np
import pandas as pd
import psutil
import requests
from scipy.stats import pearsonr, spearmanr
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
from google.colab import drive

warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 20260824
random.seed(SEED)
np.random.seed(SEED)

# ---------- Frozen model / retrieval design ----------
ENCODER_NAME = "intfloat/e5-small-v2"
ENCODER_TAG = "e5-small-v2"
QUERY_PREFIX = "query: "
PASSAGE_PREFIX = "passage: "
DIM = 384

NLIST = 4096
PQ_M = 32
PQ_NBITS = 8
REP_NPROBE = 64
SEARCH_LOW_NPROBE = 8
SEARCH_HIGH_NPROBE = 64

TOP_RETRIEVE = 100
UTILITY_K = 10
MAX_ROUNDS = 4

ALPHAS = [0.1, 0.3, 0.5, 0.7]
MEAN_K = [5, 20, 50]
SOFTMAX_K = [5, 20]
TEMPERATURES = [0.05, 0.1, 0.2, 0.5]

BOOTSTRAP_REPS = 10_000

# ---------- Execution / resource controls ----------
# First run with FULL_RUN=False. It runs a small outcome-blind implementation smoke test.
FULL_RUN = False
SMOKE_N_QUERIES = 25

# Primary v0.24 runs validation trajectories only.
# FIT is used for split definition and cheap one-shot descriptive baselines, not trajectory tuning.
RUN_FIT_TRAJECTORIES = False

CHECKPOINT_EVERY_QUERIES = 10

# Large persistent artifacts can easily consume >10 GB.
PERSIST_LARGE_ARTIFACTS = False

ENCODE_BATCH = 1024
CORPUS_BLOCK_ROWS = 100_000
INDEX_ADD_BATCH = 100_000
TRAIN_SAMPLE = 500_000

faiss.omp_set_num_threads(os.cpu_count() or 1)

print("faiss:", faiss.__version__)
print("threads:", faiss.omp_get_max_threads())
print("system RAM GiB:", round(psutil.virtual_memory().total / 2**30, 2))

In [ ]:
# Cell 2 — Mount Drive and define compact-output / large-cache roots
DRIVE_ROOT = Path("/content/drive/MyDrive")
if not DRIVE_ROOT.is_dir():
    drive.mount("/content/drive")

ARC_ROOT = DRIVE_ROOT / "rag-pq-checkpoints" / "arc-v0"
ARC_ROOT.mkdir(parents=True, exist_ok=True)

V024_ROOT = ARC_ROOT / "msmarco-external-boundary-replication-v024"
V024_ROOT.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
OUT = V024_ROOT / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)

# Raw BEIR data is kept local by default to avoid Drive quota use.
LOCAL_RAW_ROOT = Path("/content/arc-v024-raw")
LOCAL_RAW_ROOT.mkdir(parents=True, exist_ok=True)

if PERSIST_LARGE_ARTIFACTS:
    LARGE_ROOT = DRIVE_ROOT / "rag-pq-checkpoints" / "arc-v024-msmarco-large-cache"
else:
    LARGE_ROOT = Path("/content/arc-v024-large-cache")
LARGE_ROOT.mkdir(parents=True, exist_ok=True)

print("Compact output:", OUT)
print("Large artifact root:", LARGE_ROOT)
print("Persist large artifacts:", PERSIST_LARGE_ARTIFACTS)

In [ ]:
# Cell 3 — Helpers
def sha256_file(path, chunk=16 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def membership_sha(ids):
    return hashlib.sha256(
        "\n".join(sorted(map(str, ids))).encode("utf-8")
    ).hexdigest()

def iter_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield json.loads(line)

def norm_vec(x, eps=1e-12):
    x = np.asarray(x, dtype=np.float32)
    return x / max(float(np.linalg.norm(x)), eps)

def slope(y):
    y = np.asarray(y, dtype=np.float64)
    x = np.arange(len(y), dtype=np.float64)
    return float(np.polyfit(x, y, 1)[0])

def jacdist(a, b):
    A = set(map(int, a))
    B = set(map(int, b))
    return 1.0 - len(A & B) / max(1, len(A | B))

def ndcg(ids, relevant, k=UTILITY_K):
    ids = np.asarray(ids, dtype=np.int64)[:k]
    gains = np.asarray([1.0 if int(i) in relevant else 0.0 for i in ids], dtype=np.float64)
    discounts = 1.0 / np.log2(np.arange(2, len(ids) + 2))
    dcg = float(np.sum(gains * discounts))
    m = min(k, len(relevant))
    if m == 0:
        return 0.0
    idcg = float(np.sum(1.0 / np.log2(np.arange(2, m + 2))))
    return dcg / idcg

print("Helpers ready.")

## Phase A — Acquire BEIR MS MARCO without writing raw corpus into Drive

The public BEIR archive is downloaded to local Colab disk unless it is already present in this runtime.  
Only `qrels/dev.tsv` is used. No BEIR test relevance file is read by this notebook.

In [ ]:
# Cell 4 — Download / resolve BEIR MS MARCO locally
MSMARCO_DIR = LOCAL_RAW_ROOT / "msmarco"
CORPUS_JSONL = MSMARCO_DIR / "corpus.jsonl"
QUERIES_JSONL = MSMARCO_DIR / "queries.jsonl"
DEV_QRELS = MSMARCO_DIR / "qrels" / "dev.tsv"

if not (CORPUS_JSONL.is_file() and QUERIES_JSONL.is_file() and DEV_QRELS.is_file()):
    zip_path = LOCAL_RAW_ROOT / "msmarco.zip"
    if not zip_path.is_file():
        url = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/msmarco.zip"
        print("Downloading:", url)
        with requests.get(url, stream=True, timeout=120) as r:
            r.raise_for_status()
            total = int(r.headers.get("content-length", 0))
            with open(zip_path, "wb") as f, tqdm(
                total=total, unit="B", unit_scale=True, desc="msmarco.zip"
            ) as bar:
                for chunk in r.iter_content(8 * 1024 * 1024):
                    if chunk:
                        f.write(chunk)
                        bar.update(len(chunk))
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(LOCAL_RAW_ROOT)

assert CORPUS_JSONL.is_file()
assert QUERIES_JSONL.is_file()
assert DEV_QRELS.is_file()

print("MS MARCO raw data ready.")
print("corpus bytes:", CORPUS_JSONL.stat().st_size)
print("queries bytes:", QUERIES_JSONL.stat().st_size)
print("dev qrels bytes:", DEV_QRELS.stat().st_size)

In [ ]:
# Cell 5 — Load DEV qrels / query texts and create deterministic outcome-blind 50/50 split
qrels_df = pd.read_csv(DEV_QRELS, sep="\t")
# BEIR qrels normally use query-id, corpus-id, score.
q_qid_col = next(c for c in ["query-id", "query_id", "qid"] if c in qrels_df.columns)
q_doc_col = next(c for c in ["corpus-id", "corpus_id", "doc_id"] if c in qrels_df.columns)
q_score_col = next(c for c in ["score", "relevance"] if c in qrels_df.columns)

qrels_df[q_qid_col] = qrels_df[q_qid_col].astype(str)
qrels_df[q_doc_col] = qrels_df[q_doc_col].astype(str)
qrels_df[q_score_col] = qrels_df[q_score_col].astype(float)
qrels_df = qrels_df[qrels_df[q_score_col] > 0].copy()

query_text = {
    str(o["_id"]): str(o.get("text", ""))
    for o in iter_jsonl(QUERIES_JSONL)
}

DEV_QUERY_IDS = sorted(
    set(qrels_df[q_qid_col].astype(str)) & set(query_text)
)
assert DEV_QUERY_IDS, "No DEV qrels matched query texts."

# Stable hash split. No retrieval outcomes are used.
def split_key(qid):
    return hashlib.sha256(f"{SEED}|{qid}".encode("utf-8")).hexdigest()

ordered = sorted(DEV_QUERY_IDS, key=split_key)
cut = len(ordered) // 2
FIT_IDS = ordered[:cut]
VAL_IDS = ordered[cut:]

assert set(FIT_IDS).isdisjoint(VAL_IDS)
assert len(FIT_IDS) + len(VAL_IDS) == len(DEV_QUERY_IDS)

split_df = pd.DataFrame({
    "query_id": FIT_IDS + VAL_IDS,
    "split": ["fit"] * len(FIT_IDS) + ["validation"] * len(VAL_IDS),
})
SPLIT_PATH = OUT / "v024_msmarco_dev_split.csv"
split_df.to_csv(SPLIT_PATH, index=False)

print("DEV queries with positive qrels:", len(DEV_QUERY_IDS))
print("FIT:", len(FIT_IDS), "VAL:", len(VAL_IDS))
print("FIT SHA:", membership_sha(FIT_IDS))
print("VAL SHA:", membership_sha(VAL_IDS))

In [ ]:
# Cell 6 — Freeze protocol BEFORE any ANN trajectory outcome
POLICIES = []
for alpha in ALPHAS:
    for k in MEAN_K:
        POLICIES.append({
            "method": "mean",
            "alpha": float(alpha),
            "k": int(k),
            "temperature": np.nan,
            "config_key": f"mean|a={alpha}|k={k}",
        })
    for k in SOFTMAX_K:
        for tau in TEMPERATURES:
            POLICIES.append({
                "method": "softmax",
                "alpha": float(alpha),
                "k": int(k),
                "temperature": float(tau),
                "config_key": f"softmax|a={alpha}|k={k}|tau={tau}",
            })
assert len(POLICIES) == 44

pd.DataFrame(POLICIES).to_csv(OUT / "v024_frozen_policy_grid.csv", index=False)

PROTOCOL = {
    "status": "ARC_V024_MSMARCO_EXTERNAL_BOUNDARY_PROTOCOL_FROZEN_BEFORE_TRAJECTORY_OUTCOMES",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "seed": SEED,
    "dataset": "BEIR MS MARCO passage",
    "evaluation_qrels": "qrels/dev.tsv only",
    "global_pristine_claim": False,
    "trajectory_outcome_blind_claim": (
        "No prior MS MARCO ARC feedback-trajectory outcome is used to choose this protocol."
    ),
    "encoder": ENCODER_NAME,
    "dimension": DIM,
    "split": {
        "rule": "sha256(seed|query_id), sorted, deterministic 50/50",
        "n_dev": len(DEV_QUERY_IDS),
        "n_fit": len(FIT_IDS),
        "n_validation": len(VAL_IDS),
        "fit_sha256": membership_sha(FIT_IDS),
        "validation_sha256": membership_sha(VAL_IDS),
    },
    "retrievers": {
        "representation_low": {
            "type": "IndexIVFPQ",
            "nlist": NLIST,
            "m": PQ_M,
            "nbits": PQ_NBITS,
            "nprobe": REP_NPROBE,
        },
        "representation_high": {
            "type": "IndexIVFScalarQuantizer",
            "quantizer": "QT_8bit",
            "nlist": NLIST,
            "nprobe": REP_NPROBE,
        },
        "search_effort_low": {
            "type": "same IVF-SQ8",
            "nprobe": SEARCH_LOW_NPROBE,
        },
        "search_effort_high": {
            "type": "same IVF-SQ8",
            "nprobe": SEARCH_HIGH_NPROBE,
        },
    },
    "feedback": {
        "rounds": MAX_ROUNDS,
        "top_retrieve": TOP_RETRIEVE,
        "utility_k": UTILITY_K,
        "alphas": ALPHAS,
        "mean_k": MEAN_K,
        "softmax_k": SOFTMAX_K,
        "temperatures": TEMPERATURES,
        "anchoring": "q_(t+1)=normalize((1-alpha)q0 + alpha*F_t)",
        "policy_count": len(POLICIES),
    },
    "primary_endpoints": [
        "H1_query_state_slope",
        "H2_candidate_jaccard_slope",
        "H3_abs_ndcg10_gap_slope",
        "H3_signed_ndcg10_gap_slope",
    ],
    "primary_directional_hypotheses": {
        "representation_H3abs": ">0",
        "search_effort_H3abs": "<0",
    },
    "success_is_not_required": True,
    "negative_null_or_opposite_results_retained": True,
    "primary_unit": "query",
    "bootstrap_reps": BOOTSTRAP_REPS,
    "validation_trajectory_tuning_allowed": False,
    "validation_smoke_queries_used": False,
    "validation_one_shot_deferred_until_after_primary_trajectory": True,
    "controller_or_risk_model_in_scope": False,
    "test_accessed": False,
    "test_relevance_accessed": False,
}

PROTOCOL_PATH = OUT / "v024_frozen_protocol.json"
PROTOCOL_PATH.write_text(json.dumps(PROTOCOL, indent=2, sort_keys=True))
PROTOCOL_SHA = sha256_file(PROTOCOL_PATH)

(OUT / "V024_PROTOCOL_SHA256.txt").write_text(
    f"{PROTOCOL_SHA}  {PROTOCOL_PATH.name}\n"
)

print("Protocol SHA:", PROTOCOL_SHA)
print("PROTOCOL FROZEN — PASS")

## Phase B — Encode E5 queries and full MS MARCO corpus

The full corpus is encoded in resumable 100k-document blocks.  
Each block is a single float16 `.npy` artifact; no second full corpus memmap is created, which avoids doubling storage.

When `PERSIST_LARGE_ARTIFACTS=False`, these blocks live in `/content` and disappear if the Colab runtime is destroyed.

In [ ]:
# Cell 7 — Query encoding
model = SentenceTransformer(ENCODER_NAME)

QUERY_IDS_PATH = LARGE_ROOT / "dev_query_ids.txt"
QUERY_EMB_PATH = LARGE_ROOT / "dev_query_embeddings.float32.npy"

if QUERY_EMB_PATH.is_file() and QUERY_IDS_PATH.is_file():
    dev_query_embeddings = np.load(QUERY_EMB_PATH)
    assert QUERY_IDS_PATH.read_text().splitlines() == DEV_QUERY_IDS
else:
    texts = [QUERY_PREFIX + query_text[qid] for qid in DEV_QUERY_IDS]
    dev_query_embeddings = model.encode(
        texts,
        batch_size=ENCODE_BATCH,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    ).astype(np.float32)
    np.save(QUERY_EMB_PATH, dev_query_embeddings)
    QUERY_IDS_PATH.write_text("\n".join(DEV_QUERY_IDS) + "\n")

assert dev_query_embeddings.shape == (len(DEV_QUERY_IDS), DIM)
DEV_QUERY_INDEX = {qid: i for i, qid in enumerate(DEV_QUERY_IDS)}
print("Query embeddings:", dev_query_embeddings.shape)

In [ ]:
# Cell 8 — Encode corpus in resumable blocks and map only qrels-relevant docs to rows
BLOCK_DIR = LARGE_ROOT / "corpus_blocks"
BLOCK_DIR.mkdir(parents=True, exist_ok=True)

RELEVANT_DOC_IDS = set(qrels_df[q_doc_col].astype(str))
QREL_ROW_MAP_PATH = LARGE_ROOT / "dev_qrel_doc_rows.csv"
CORPUS_MANIFEST_PATH = LARGE_ROOT / "corpus_block_manifest.json"

block_records = []
relevant_row_map = {}

ids_buf, texts_buf = [], []
block_idx = 0
global_row = 0

def save_or_reuse_block(idx, texts, row_start):
    ep = BLOCK_DIR / f"block-{idx:04d}.float16.npy"
    if ep.is_file():
        arr = np.load(ep, mmap_mode="r")
        assert arr.shape == (len(texts), DIM), (ep, arr.shape, len(texts))
        return {
            "block": idx,
            "row_start": row_start,
            "row_end": row_start + len(texts),
            "rows": len(texts),
            "path": str(ep),
            "sha256": sha256_file(ep),
            "reused": True,
        }

    vec = model.encode(
        [PASSAGE_PREFIX + t for t in texts],
        batch_size=ENCODE_BATCH,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    ).astype(np.float16)
    np.save(ep, vec)
    return {
        "block": idx,
        "row_start": row_start,
        "row_end": row_start + len(texts),
        "rows": len(texts),
        "path": str(ep),
        "sha256": sha256_file(ep),
        "reused": False,
    }

t0 = time.perf_counter()

for obj in iter_jsonl(CORPUS_JSONL):
    doc_id = str(obj["_id"])
    text = str(obj.get("text", ""))
    title = str(obj.get("title", ""))
    full_text = (title + " " + text).strip() if title else text

    if doc_id in RELEVANT_DOC_IDS:
        relevant_row_map[doc_id] = global_row

    ids_buf.append(doc_id)
    texts_buf.append(full_text)
    global_row += 1

    if len(texts_buf) >= CORPUS_BLOCK_ROWS:
        row_start = global_row - len(texts_buf)
        block_records.append(save_or_reuse_block(block_idx, texts_buf, row_start))
        ids_buf, texts_buf = [], []
        block_idx += 1

if texts_buf:
    row_start = global_row - len(texts_buf)
    block_records.append(save_or_reuse_block(block_idx, texts_buf, row_start))

N_DOCS = global_row
assert sum(r["rows"] for r in block_records) == N_DOCS

missing_rel_docs = sorted(RELEVANT_DOC_IDS - set(relevant_row_map))
assert not missing_rel_docs, f"Missing relevant docs: {missing_rel_docs[:10]}"

pd.DataFrame(
    [{"doc_id": d, "corpus_row": r} for d, r in relevant_row_map.items()]
).to_csv(QREL_ROW_MAP_PATH, index=False)

corpus_manifest = {
    "status": "V024_MSMARCO_E5_CORPUS_BLOCKS_COMPLETE",
    "encoder": ENCODER_NAME,
    "dimension": DIM,
    "corpus_rows": N_DOCS,
    "block_rows": CORPUS_BLOCK_ROWS,
    "blocks": block_records,
    "qrels_relevant_docs_mapped": len(relevant_row_map),
}
CORPUS_MANIFEST_PATH.write_text(json.dumps(corpus_manifest, indent=2))

print("Corpus rows:", f"{N_DOCS:,}")
print("Blocks:", len(block_records))
print("Relevant docs mapped:", len(relevant_row_map))
print("Encoding/scan seconds:", round(time.perf_counter() - t0, 1))

In [ ]:
# Cell 9 — Open block memmaps + row lookup helpers
blocks = []
block_starts = []
block_ends = []

for r in block_records:
    arr = np.load(r["path"], mmap_mode="r")
    assert arr.shape == (r["rows"], DIM)
    blocks.append(arr)
    block_starts.append(int(r["row_start"]))
    block_ends.append(int(r["row_end"]))

block_starts = np.asarray(block_starts, dtype=np.int64)
block_ends = np.asarray(block_ends, dtype=np.int64)

def fetch_docs(rows):
    rows = np.asarray(rows, dtype=np.int64)
    out = np.empty((len(rows), DIM), dtype=np.float32)
    # Number of blocks is small (~89), so a simple grouped lookup is adequate.
    bidx = np.searchsorted(block_ends, rows, side="right")
    for b in np.unique(bidx):
        mask = bidx == b
        local = rows[mask] - block_starts[b]
        out[mask] = np.asarray(blocks[b][local], dtype=np.float32)
    out /= np.maximum(np.linalg.norm(out, axis=1, keepdims=True), 1e-12)
    return out

# Build qrels in corpus-row space.
doc_to_row = {str(k): int(v) for k, v in relevant_row_map.items()}
QRELS = defaultdict(set)
for _, r in qrels_df.iterrows():
    qid = str(r[q_qid_col])
    docid = str(r[q_doc_col])
    if qid in DEV_QUERY_INDEX and docid in doc_to_row:
        QRELS[qid].add(doc_to_row[docid])

missing_qrels = [qid for qid in DEV_QUERY_IDS if not QRELS[qid]]
assert not missing_qrels, f"Missing mapped qrels for {len(missing_qrels)} queries"

# Semantic sanity check: relevant document vectors must be finite and nonzero.
sample_rel_rows = np.array(list(doc_to_row.values())[: min(1000, len(doc_to_row))], dtype=np.int64)
sample_rel_vecs = fetch_docs(sample_rel_rows)
assert np.isfinite(sample_rel_vecs).all()
assert (np.linalg.norm(sample_rel_vecs, axis=1) > 0.99).all()

print("Block lookup + qrels row mapping — PASS")

## Phase C — Build two E5 IVF indexes

Both indexes are trained from the same deterministic block-level sample pool.  
This is not an exact-oracle experiment; SQ8 is a relative higher-representation-fidelity comparator, matching the current paper's interpretation.

In [ ]:
# Cell 10 — Deterministic training sample
TRAIN_PATH = LARGE_ROOT / f"train_sample_{TRAIN_SAMPLE}.float32.npy"

if TRAIN_PATH.is_file():
    train = np.load(TRAIN_PATH)
else:
    rng_train = np.random.default_rng(SEED + 240)
    per_block = int(math.ceil(TRAIN_SAMPLE / len(blocks)))
    pieces = []
    for arr in blocks:
        n = len(arr)
        take = min(per_block, n)
        idx = rng_train.choice(n, size=take, replace=False)
        pieces.append(np.asarray(arr[idx], dtype=np.float32))
    train = np.concatenate(pieces, axis=0)[:TRAIN_SAMPLE]
    train /= np.maximum(np.linalg.norm(train, axis=1, keepdims=True), 1e-12)
    np.save(TRAIN_PATH, train)

assert train.ndim == 2 and train.shape[1] == DIM
print("Training sample:", train.shape)

In [ ]:
# Cell 11 — Build / load IVF-PQ32 and IVF-SQ8
PQ32_PATH = LARGE_ROOT / "msmarco-e5-ivfpq-nlist4096-m32-nbits8.faiss"
SQ8_PATH = LARGE_ROOT / "msmarco-e5-ivfsq8-nlist4096.faiss"

def stream_add(index):
    added = 0
    for arr in tqdm(blocks, desc="adding corpus blocks"):
        x = np.asarray(arr, dtype=np.float32)
        x /= np.maximum(np.linalg.norm(x, axis=1, keepdims=True), 1e-12)
        index.add(x)
        added += len(x)
    return added

if PQ32_PATH.is_file():
    pq32 = faiss.read_index(str(PQ32_PATH))
else:
    q = faiss.IndexFlatIP(DIM)
    pq32 = faiss.IndexIVFPQ(
        q, DIM, NLIST, PQ_M, PQ_NBITS, faiss.METRIC_INNER_PRODUCT
    )
    pq32.train(train)
    assert pq32.is_trained
    assert stream_add(pq32) == N_DOCS
    faiss.write_index(pq32, str(PQ32_PATH))

if SQ8_PATH.is_file():
    sq8 = faiss.read_index(str(SQ8_PATH))
else:
    q = faiss.IndexFlatIP(DIM)
    sq8 = faiss.IndexIVFScalarQuantizer(
        q,
        DIM,
        NLIST,
        faiss.ScalarQuantizer.QT_8bit,
        faiss.METRIC_INNER_PRODUCT,
    )
    sq8.train(train)
    assert sq8.is_trained
    assert stream_add(sq8) == N_DOCS
    faiss.write_index(sq8, str(SQ8_PATH))

assert pq32.ntotal == sq8.ntotal == N_DOCS

print("PQ32 ntotal:", pq32.ntotal)
print("SQ8 ntotal:", sq8.ntotal)
print("PQ32 size GiB:", round(PQ32_PATH.stat().st_size / 2**30, 3))
print("SQ8 size GiB:", round(SQ8_PATH.stat().st_size / 2**30, 3))
print("INDEX BUILD/LOAD — PASS")

In [ ]:
# Cell 12 — FIT-only one-shot descriptive baselines
# IMPORTANT: validation one-shot evaluation is deliberately deferred until
# AFTER the complete primary validation trajectory sweep and frozen gate.

def search_index(index, q, nprobe):
    index.nprobe = int(nprobe)
    q = norm_vec(q)[None, :]
    s, I = index.search(q, TOP_RETRIEVE)
    valid = I[0] >= 0
    return s[0][valid], I[0][valid]

def eval_one_shot(qids, label):
    rows = []
    for qid in tqdm(qids, desc=f"one-shot {label}"):
        q = dev_query_embeddings[DEV_QUERY_INDEX[qid]]
        _, i_pq = search_index(pq32, q, REP_NPROBE)
        _, i_sq8_8 = search_index(sq8, q, SEARCH_LOW_NPROBE)
        _, i_sq8_64 = search_index(sq8, q, SEARCH_HIGH_NPROBE)
        rel = QRELS[qid]
        rows.append({
            "query_id": qid,
            "split": label,
            "pq32_n64": ndcg(i_pq, rel),
            "sq8_n8": ndcg(i_sq8_8, rel),
            "sq8_n64": ndcg(i_sq8_64, rel),
        })
    return pd.DataFrame(rows)

ONE_SHOT_PATH = OUT / "v024_one_shot_ndcg10.csv"

one_fit = eval_one_shot(FIT_IDS, "fit")
one = one_fit.copy()
one.to_csv(ONE_SHOT_PATH, index=False)

display(
    one.groupby("split")[["pq32_n64", "sq8_n8", "sq8_n64"]]
    .mean()
    .round(6)
)

VALIDATION_ONE_SHOT_COMPLETE = False

print("FIT one-shot descriptive baseline — COMPLETE")
print("Validation one-shot evaluation is DEFERRED until after the primary trajectory gate.")

In [ ]:
# Cell 13 — Unified search + feedback helpers
MECHANISMS = ["representation", "nprobe"]

def retrieve(mechanism, fidelity, q):
    if mechanism == "representation":
        if fidelity == "L":
            return search_index(pq32, q, REP_NPROBE)
        return search_index(sq8, q, REP_NPROBE)

    if mechanism == "nprobe":
        if fidelity == "L":
            return search_index(sq8, q, SEARCH_LOW_NPROBE)
        return search_index(sq8, q, SEARCH_HIGH_NPROBE)

    raise ValueError(mechanism)

def feedback_vector(scores, ids, policy):
    k = int(policy["k"])
    ids = np.asarray(ids[:k], dtype=np.int64)
    scores = np.asarray(scores[:k], dtype=np.float64)
    docs = fetch_docs(ids)

    if policy["method"] == "mean":
        f = docs.mean(axis=0)
    else:
        tau = float(policy["temperature"])
        z = scores / tau
        z -= z.max()
        w = np.exp(z)
        w /= w.sum()
        f = (docs * w[:, None]).sum(axis=0)

    return norm_vec(f)

def update_state(q0, feedback, alpha):
    return norm_vec((1.0 - float(alpha)) * q0 + float(alpha) * feedback)

print("Trajectory helpers ready.")

In [ ]:
# Cell 14 — Exact v0.18-style coupled trajectory implementation
def run_pair(qid, policy, mechanism):
    q0 = norm_vec(dev_query_embeddings[DEV_QUERY_INDEX[qid]])
    qL = q0.copy()
    qH = q0.copy()
    rel = QRELS[qid]
    rows = []

    for t in range(MAX_ROUNDS + 1):
        sL, iL = retrieve(mechanism, "L", qL)
        sH, iH = retrieve(mechanism, "H", qH)

        if min(len(iL), len(iH)) < max(int(policy["k"]), UTILITY_K):
            raise RuntimeError(
                f"Insufficient retrieval results: qid={qid}, mechanism={mechanism}, policy={policy}"
            )

        uL = ndcg(iL, rel)
        uH = ndcg(iH, rel)

        rows.append({
            "query_id": str(qid),
            "mechanism": mechanism,
            "iteration": int(t),
            "method": policy["method"],
            "alpha": float(policy["alpha"]),
            "k": int(policy["k"]),
            "temperature": (
                float(policy["temperature"])
                if not pd.isna(policy["temperature"]) else np.nan
            ),
            "config_key": policy["config_key"],
            "query_state_distance": 1.0 - float(np.dot(norm_vec(qL), norm_vec(qH))),
            "candidate_jaccard_distance": jacdist(
                iL[:TOP_RETRIEVE], iH[:TOP_RETRIEVE]
            ),
            "utility_low": uL,
            "utility_high": uH,
            "signed_utility_gap": uH - uL,
            "abs_utility_gap": abs(uH - uL),
        })

        if t == MAX_ROUNDS:
            break

        fL = feedback_vector(sL, iL, policy)
        fH = feedback_vector(sH, iH, policy)

        qL = update_state(q0, fL, policy["alpha"])
        qH = update_state(q0, fH, policy["alpha"])

    return rows

def endpoint_df(traj):
    out = []
    group_cols = [
        "query_id", "mechanism", "method", "alpha", "k",
        "temperature", "config_key"
    ]
    for keys, g in traj.groupby(group_cols, dropna=False, sort=False):
        g = g.sort_values("iteration")
        out.append({
            "query_id": keys[0],
            "mechanism": keys[1],
            "method": keys[2],
            "alpha": keys[3],
            "k": keys[4],
            "temperature": keys[5],
            "config_key": keys[6],
            "H1_slope": slope(g["query_state_distance"]),
            "H2_slope": slope(g["candidate_jaccard_distance"]),
            "H3_abs_slope": slope(g["abs_utility_gap"]),
            "H3_signed_slope": slope(g["signed_utility_gap"]),
            "final_signed_gap": float(g["signed_utility_gap"].iloc[-1]),
        })
    return pd.DataFrame(out)

print("Coupled trajectory implementation ready.")

In [ ]:
# Cell 15 — FIT-only implementation smoke test
# This cell MUST NOT use validation queries.
smoke_qids = FIT_IDS[: min(SMOKE_N_QUERIES, len(FIT_IDS))]
smoke_policy = POLICIES[0]

assert set(smoke_qids).isdisjoint(set(VAL_IDS))

for mech in MECHANISMS:
    s = pd.DataFrame(run_pair(smoke_qids[0], smoke_policy, mech))
    assert len(s) == MAX_ROUNDS + 1
    assert np.isfinite(s[[
        "query_state_distance",
        "candidate_jaccard_distance",
        "utility_low",
        "utility_high",
        "signed_utility_gap",
        "abs_utility_gap",
    ]].to_numpy()).all()
    display(s[[
        "mechanism", "iteration", "query_state_distance",
        "candidate_jaccard_distance", "abs_utility_gap", "signed_utility_gap"
    ]])

print("FIT-ONLY IMPLEMENTATION SMOKE TEST — PASS")
print("No validation trajectory has been inspected by this smoke test.")
print("Do not change contrasts, split, policies, endpoints, or thresholds based on FIT smoke values.")

## Before the expensive primary run

After Cell 15 passes:

1. **Do not rerun Cell 2 or Cell 6.** Keep the same `RUN_ID`, `OUT`, split, and frozen protocol.
2. Do not change any frozen scientific constant.
3. In a new small code cell, execute only:

```python
FULL_RUN = True
print("FULL_RUN =", FULL_RUN)
```

4. Then run **Cell 16 → Cell 20**.
5. Only after Cell 20 prints the frozen directional gate, run **Cell 20.1** to evaluate validation one-shot effectiveness.
6. Finally run Cell 21 to write the report and hashes.

The primary trajectory analysis uses the 3,490 validation queries only.  
The smoke test uses FIT queries only.  
Validation one-shot diagnostics are intentionally post-primary.

In [ ]:
# Cell 16 — Resumable primary trajectory sweep
if FULL_RUN:
    PRIMARY_IDS = VAL_IDS
    mode = "validation-full"
else:
    PRIMARY_IDS = smoke_qids
    mode = f"smoke-{len(PRIMARY_IDS)}"
    print("WARNING: FULL_RUN=False, so this is only a FIT smoke sweep, not the primary validation analysis.")

RUN_DIR = OUT / mode
RUN_DIR.mkdir(parents=True, exist_ok=True)

print("mode:", mode)
print("queries:", len(PRIMARY_IDS))
print("mechanisms:", MECHANISMS)
print("policies:", len(POLICIES))
print("expected trajectory rows:",
      len(PRIMARY_IDS) * len(MECHANISMS) * len(POLICIES) * (MAX_ROUNDS + 1))

for start in range(0, len(PRIMARY_IDS), CHECKPOINT_EVERY_QUERIES):
    stop = min(start + CHECKPOINT_EVERY_QUERIES, len(PRIMARY_IDS))
    cp = RUN_DIR / f"traj_{start:05d}_{stop:05d}.parquet"

    if cp.exists():
        print("skip", cp.name)
        continue

    rows = []
    t0 = time.perf_counter()

    for qi in range(start, stop):
        qid = PRIMARY_IDS[qi]
        for mechanism in MECHANISMS:
            for policy in POLICIES:
                rows.extend(run_pair(qid, policy, mechanism))

    df_cp = pd.DataFrame(rows)
    tmp = cp.with_suffix(".tmp.parquet")
    df_cp.to_parquet(tmp, index=False)
    os.replace(tmp, cp)

    dt = time.perf_counter() - t0
    print(f"wrote {cp.name}: {len(df_cp):,} rows | {dt:.1f}s")

parts = sorted(RUN_DIR.glob("traj_*.parquet"))
assert parts

traj = pd.concat([pd.read_parquet(p) for p in parts], ignore_index=True)

expected = (
    len(PRIMARY_IDS)
    * len(MECHANISMS)
    * len(POLICIES)
    * (MAX_ROUNDS + 1)
)
assert len(traj) == expected, (len(traj), expected)
assert traj["query_id"].nunique() == len(PRIMARY_IDS)

TRAJ_PATH = OUT / f"v024_{mode}_trajectories.parquet"
traj.to_parquet(TRAJ_PATH, index=False)

endpoints = endpoint_df(traj)
ENDPOINT_PATH = OUT / f"v024_{mode}_endpoints.parquet"
endpoints.to_parquet(ENDPOINT_PATH, index=False)

print("Trajectory rows:", f"{len(traj):,}")
print("Endpoint rows:", f"{len(endpoints):,}")
print("PRIMARY SWEEP COMPLETE")

In [ ]:
# Cell 17 — Query-level aggregation
MEASURES = ["H1_slope", "H2_slope", "H3_abs_slope", "H3_signed_slope"]

query_mech = (
    endpoints.groupby(["query_id", "mechanism"], as_index=False)[MEASURES]
    .mean()
)

summary_point = query_mech.groupby("mechanism")[MEASURES].mean()
display(summary_point.round(6))

In [ ]:
# Cell 18 — Query-cluster bootstrap confidence intervals
rng = np.random.default_rng(SEED + 2401)

def bootstrap_mean_ci(series, reps=BOOTSTRAP_REPS):
    x = pd.Series(series).dropna().to_numpy(np.float64)
    n = len(x)
    assert n > 0
    point = float(x.mean())

    boots = np.empty(reps, dtype=np.float64)
    for b in range(reps):
        idx = rng.integers(0, n, size=n)
        boots[b] = float(x[idx].mean())

    lo, hi = np.quantile(boots, [0.025, 0.975])
    return point, float(lo), float(hi), n

summary_rows = []
for mech in MECHANISMS:
    sub = query_mech[query_mech["mechanism"].eq(mech)]
    for measure in MEASURES:
        point, lo, hi, n = bootstrap_mean_ci(sub[measure])
        summary_rows.append({
            "mechanism": mech,
            "measure": measure,
            "mean": point,
            "ci95_low": lo,
            "ci95_high": hi,
            "n_queries": n,
        })

summary = pd.DataFrame(summary_rows)
SUMMARY_PATH = OUT / f"v024_{mode}_query_bootstrap_summary.csv"
summary.to_csv(SUMMARY_PATH, index=False)

display(summary.round(6))

In [ ]:
# Cell 19 — Regime / signed-direction descriptive audit
EPS_PRIMARY = 0.002

regime_rows = []
for mech in MECHANISMS:
    sub = endpoints[endpoints["mechanism"].eq(mech)].copy()
    h = sub["H3_abs_slope"].to_numpy(np.float64)

    stable = np.abs(h) <= EPS_PRIMARY
    amp = h > EPS_PRIMARY
    contract = h < -EPS_PRIMARY

    amp_sub = sub.loc[amp]
    if len(amp_sub):
        higher_win = float((amp_sub["final_signed_gap"] > 0).mean())
        lower_win = float((amp_sub["final_signed_gap"] < 0).mean())
        tie = float((amp_sub["final_signed_gap"] == 0).mean())
    else:
        higher_win = lower_win = tie = np.nan

    regime_rows.append({
        "mechanism": mech,
        "epsilon": EPS_PRIMARY,
        "stable_null_fraction": float(stable.mean()),
        "amplifying_fraction": float(amp.mean()),
        "contracting_fraction": float(contract.mean()),
        "amplification_events": int(amp.sum()),
        "higher_fidelity_win_given_amplification": higher_win,
        "lower_fidelity_win_given_amplification": lower_win,
        "tie_given_amplification": tie,
    })

regimes = pd.DataFrame(regime_rows)
REGIME_PATH = OUT / f"v024_{mode}_regime_summary.csv"
regimes.to_csv(REGIME_PATH, index=False)
display(regimes.round(6))

In [ ]:
# Cell 20 — Frozen directional gate
def ci_for(mech, measure):
    r = summary[
        summary["mechanism"].eq(mech)
        & summary["measure"].eq(measure)
    ].iloc[0]
    return float(r["mean"]), float(r["ci95_low"]), float(r["ci95_high"])

rep_h3 = ci_for("representation", "H3_abs_slope")
np_h3 = ci_for("nprobe", "H3_abs_slope")

gate = {
    "status": "ARC_V024_EXTERNAL_BOUNDARY_ANALYZED",
    "mode": mode,
    "n_queries": len(PRIMARY_IDS),
    "representation_H3abs": {
        "mean": rep_h3[0],
        "ci95": [rep_h3[1], rep_h3[2]],
        "hypothesis": ">0",
        "direction_matches": bool(rep_h3[0] > 0),
    },
    "nprobe_H3abs": {
        "mean": np_h3[0],
        "ci95": [np_h3[1], np_h3[2]],
        "hypothesis": "<0",
        "direction_matches": bool(np_h3[0] < 0),
    },
    "boundary_sign_map_matches_current_paper": bool(
        rep_h3[0] > 0 and np_h3[0] < 0
    ),
    "negative_null_or_opposite_results_retained": True,
    "validation_trajectory_tuning_performed": False,
    "test_accessed": False,
    "test_relevance_accessed": False,
}

print(json.dumps(gate, indent=2))

In [ ]:
# Cell 20.1 — Post-primary validation one-shot descriptive evaluation
# RUN ONLY AFTER:
#   (a) FULL_RUN=True,
#   (b) Cell 16 completed the full validation trajectory sweep,
#   (c) Cells 17–20 completed the frozen primary analysis/gate.

assert FULL_RUN is True, "Do not run validation one-shot before the full primary trajectory analysis."
assert mode == "validation-full", mode
assert "gate" in globals(), "Run Cell 20 first."
assert int(gate["n_queries"]) == len(VAL_IDS), gate

# Require the persisted full primary trajectory and endpoint artifacts.
assert TRAJ_PATH.is_file(), TRAJ_PATH
assert ENDPOINT_PATH.is_file(), ENDPOINT_PATH
assert SUMMARY_PATH.is_file(), SUMMARY_PATH
assert REGIME_PATH.is_file(), REGIME_PATH

one_val = eval_one_shot(VAL_IDS, "validation")
one = pd.concat([one_fit, one_val], ignore_index=True)
one.to_csv(ONE_SHOT_PATH, index=False)

VALIDATION_ONE_SHOT_COMPLETE = True

display(
    one.groupby("split")[["pq32_n64", "sq8_n8", "sq8_n64"]]
    .mean()
    .round(6)
)

print("POST-PRIMARY VALIDATION ONE-SHOT — COMPLETE")
print("Primary trajectory results were already persisted before this evaluation.")


In [ ]:
# Cell 21 — Final report + hashes
assert FULL_RUN is True, "Final report is for the primary validation run."
assert VALIDATION_ONE_SHOT_COMPLETE is True, "Run Cell 20.1 after the primary gate before finalizing."
REPORT = {
    **gate,
    "protocol_sha256": PROTOCOL_SHA,
    "dataset": "BEIR MS MARCO passage",
    "encoder": ENCODER_NAME,
    "corpus_rows": int(N_DOCS),
    "split_path": str(SPLIT_PATH),
    "one_shot_path": str(ONE_SHOT_PATH),
    "trajectory_path": str(TRAJ_PATH),
    "endpoint_path": str(ENDPOINT_PATH),
    "summary_path": str(SUMMARY_PATH),
    "regime_path": str(REGIME_PATH),
    "large_artifacts_persisted_to_drive": bool(PERSIST_LARGE_ARTIFACTS),
    "validation_smoke_queries_used": False,
    "validation_one_shot_timing": "post-primary trajectory and frozen gate",
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
}

REPORT_PATH = OUT / f"v024_{mode}_final_report.json"
REPORT_PATH.write_text(json.dumps(REPORT, indent=2))

artifact_paths = [
    PROTOCOL_PATH,
    SPLIT_PATH,
    OUT / "v024_frozen_policy_grid.csv",
    ONE_SHOT_PATH,
    TRAJ_PATH,
    ENDPOINT_PATH,
    SUMMARY_PATH,
    REGIME_PATH,
    REPORT_PATH,
]

hash_rows = []
for p in artifact_paths:
    if p.exists():
        hash_rows.append({
            "file": p.name,
            "bytes": p.stat().st_size,
            "sha256": sha256_file(p),
        })

hash_df = pd.DataFrame(hash_rows)
HASH_PATH = OUT / f"V024_{mode.upper().replace('-', '_')}_ARTIFACT_SHA256.csv"
hash_df.to_csv(HASH_PATH, index=False)

print("=" * 88)
print("ARC-v0.24 MS MARCO EXTERNAL BOUNDARY REPLICATION — COMPLETE")
print("mode:", mode)
print("output:", OUT)
print("boundary sign map matches:", gate["boundary_sign_map_matches_current_paper"])
print("test accessed:", False)
print("=" * 88)
display(hash_df)

## Optional phase — FIT trajectories

Do **not** enable this to rescue, tune, reinterpret, or select a validation result.

If the primary validation run is complete and Compute Units remain, a separate follow-up may characterize FIT↔validation policy-structure reproducibility. It is not required for the ARC-v0.24 primary external-boundary result.

## Safe execution order

1. Cells 1–15 with `FULL_RUN=False`.
2. Verify Cell 15 says `FIT-ONLY IMPLEMENTATION SMOKE TEST — PASS`.
3. Execute `FULL_RUN = True` in a small new cell.
4. Run Cells 16–20.
5. Run Cell 20.1 **only after** Cell 20 has produced the frozen gate.
6. Run Cell 21.

Do **not** rerun Cell 2 or Cell 6 between the smoke test and the primary validation run.

## What to send back

After the full run, send the executed notebook or:

- `v024_validation-full_query_bootstrap_summary.csv`
- `v024_validation-full_regime_summary.csv`
- `v024_one_shot_ndcg10.csv`
- `v024_validation-full_final_report.json`

The two primary quantities are:

- `representation / H3_abs_slope`
- `nprobe / H3_abs_slope`

Do not alter nprobe values, representation contrast, policy grid, split, endpoint definitions, or thresholds after seeing validation trajectories.